<p align="center">
<img src="Images/sorbonne_logo.png" alt="Logo" width="300"/>
</p>

# **Module 1 - Data Extraction & Manipulation**

* **Author**: Elia Landini
* **Student ID**: 12310239
* **Course**: EESM2-Financial Economics 
* **Supervisor**: Jean-Bernard Chatelain
* **Reference Repository**: https://github.com/EliaLand/Policy_Target_RegimeSwitchingPersistence

### **1) PREFACE**

abcd

### **2) REQUIREMENTS SET-UP**

In [90]:
# Requirements.txt file installation
#!pip install -r requirements.txt

In [91]:
# Libraries import
import warnings
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import scipy.stats as stats
from scipy.stats import norm
from scipy.stats import levene
from scipy.stats import ks_2samp
from scipy.stats import kstest
from scipy.stats import pearsonr
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.tsa.stattools import adfuller
import sklearn.tree
import sklearn.metrics
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing 
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, roc_curve, confusion_matrix,
                             precision_score, recall_score, f1_score,
                             accuracy_score, precision_recall_curve, auc, 
                             RocCurveDisplay, ConfusionMatrixDisplay)
from sklearn.linear_model import (LinearRegression, LogisticRegression)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.utils.class_weight import compute_class_weight
import plotly.express as px
from stargazer.stargazer import Stargazer
from IPython.core.display import HTML
from IPython.display import Image
import itertools
from arch.unitroot import PhillipsPerron

### **3) HELPER FUNCTIONS & GENERAL VARIABLES**

In [92]:
# Custom modules import
from Modules.FRED_module import fetch_FRED
from Modules.EUROSTAT_module import fetch_EUROSTAT
from Modules.WB_module import fetch_WB 
from Modules.YFINANCE_module import fetch_YFINANCE
from Modules.DBNOMICS_module import fetch_DBNOMICS

In [93]:
# Statistical Significance labelling 
def significance_stars(p):
    if p < 0.001:
        return "***"  
    elif p < 0.01:
        return "**"    
    elif p < 0.05:
        return "*"   
    else:
        return ""

In [94]:
# We supress potential warnings with this command
warnings.filterwarnings("ignore")

### **4) DATA RETRIEVAL**

##### **4.1) REAL EFFECTIVE EXCHANGE RATES**

In [95]:
# Real Effective Exchange Rates: CPI Based for Japan (monthly, Index 2015=100, non-seasonally adjusted, 1970-01, 2025-09)
# https://fred.stlouisfed.org/series/CCRETT01JPM661N

REXUSDJPY_m_raw = fetch_FRED("CCRETT01JPM661N") 
REXUSDJPY_m_raw = REXUSDJPY_m_raw.rename(columns= 
        {"date": "Time", 
         "CCRETT01JPM661N": "USD-JPY reer CPI-based (Index 2015=100)"
})

REXUSDJPY_m_raw["Time"] = REXUSDJPY_m_raw["Time"].dt.to_period("M").astype(str)

REXUSDJPY_m_raw["Country"] = "JP"
REXUSDJPY_m_raw = REXUSDJPY_m_raw[["Country", "Time", "USD-JPY reer CPI-based (Index 2015=100)"]]

REXUSDJPY_m_raw.tail()

,Country,Time,USD-JPY reer CPI-based (Index 2015=100)
668,JP,2025-09,80.85533
669,JP,2025-10,79.29526
670,JP,2025-11,77.64572
671,JP,2025-12,76.62940
672,JP,2026-01,75.62581


##### **4.2) CONSUMER PRICES (HICP), ALL ITEMS (WHOLE COUNTRY), NSA**

In [96]:
# Consumer prices (HICP), all items (whole country), NSA, Japan (monthly, Index April 2025=111.5, non-seasonally adjusted, 1970-01, 2025-04)
# https://data.ecb.europa.eu/data/datasets/RTD/RTD.M.JP.N.P_C_OV.X

jp_HICP_m_raw = pd.read_csv("Data/HICP_ECB_extracted_raw.csv") 
jp_HICP_m_raw = jp_HICP_m_raw.drop(columns=["DATE"])
jp_HICP_m_raw = jp_HICP_m_raw.rename(columns= 
        {"TIME PERIOD": "Time", 
         "CONSUMER PRICES, ALL ITEMS (WHOLE COUNTRY), NSA (RTD.M.JP.N.P_C_OV.X)": "HICP (NSA)"
})     
                          
jp_HICP_m_raw["Time"] = pd.to_datetime(jp_HICP_m_raw["Time"], format="%Y%b", errors="coerce")
jp_HICP_m_raw["Time"] = jp_HICP_m_raw["Time"].dt.to_period("M").astype(str)

jp_HICP_m_raw["Country"] = "JP"
jp_HICP_m_raw = jp_HICP_m_raw[["Country", "Time", "HICP (NSA)"]]

jp_HICP_m_raw.tail()

,Country,Time,HICP (NSA)
659,JP,2024-12,110.7
660,JP,2025-01,111.2
661,JP,2025-02,110.8
662,JP,2025-03,111.1
663,JP,2025-04,111.5


##### **4.3) JPY-USD SPOT EXCHANGE RATE**

In [97]:
# Japanese Yen to U.S. Dollar Spot Exchange Rate (monthly, non-seasonally adjusted, 1971-01, 2025-09)
# https://fred.stlouisfed.org/series/EXJPUS

JPYUSD_m_raw = fetch_FRED("EXJPUS") 
JPYUSD_m_raw = JPYUSD_m_raw.rename(columns= 
        {"date": "Time", 
         "EXJPUS": "JPY-USD Spot Exchange Rate"
})

JPYUSD_m_raw["Time"] = JPYUSD_m_raw["Time"].dt.to_period("M").astype(str)

JPYUSD_m_raw["Country"] = "JP"
JPYUSD_m_raw = JPYUSD_m_raw[["Country", "Time", "JPY-USD Spot Exchange Rate"]]

JPYUSD_m_raw.tail()

,Country,Time,JPY-USD Spot Exchange Rate
656,JP,2025-09,147.8629
657,JP,2025-10,151.3545
658,JP,2025-11,155.1411
659,JP,2025-12,155.9150
660,JP,2026-01,156.6505


##### **4.4) STOCK INDEX/BOND-RELATED INSTRUMENTS**

In [98]:
# Monthly price and volume of Japan's stock indeces and bond-related instruments (monthly, log, number of securities traded, 2015-01, 2025-10)
# (!!!) Volume column is dangerous, a lot of 0 values, depending on the index, it must be carefully handled 

jp_stock_indices_tickers = {
    "Nikkei 225": "^N225",
    "NYSE Arca Japan Index": "^JPN",
    "iShares 7‑10 Year Japan Government Bond ETF": "236A.T",
    "iShares Core Japan Government Bond ETF": "2561.T"
}

start = "2015-01-01"
end = "2025-10-25"

list_single_country_dfs = []

# We iterate over each country and respective stock index
# We aggregate data through concatenation based on y axis
for index, ticker in jp_stock_indices_tickers.items():
    df = fetch_YFINANCE(ticker, start, end)

# MultiIndex columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] for col in df.columns]
    df = df.rename(columns={
        "Close": "Index-specific Closing Price",
        "YearMonth": "Time"
    })
    df["Log Monthly Return"] = np.log(df["Index-specific Closing Price"] / df["Index-specific Closing Price"].shift(1))
    df["Stock Index / Bond-related Instrument"] = index
    df["Stock Index"] = ticker

    df = df[["Stock Index / Bond-related Instrument", "Stock Index", "Time", "Log Monthly Return", "Volume"]]

    list_single_country_dfs.append(df)

jp_stock_m_raw = pd.concat(list_single_country_dfs, ignore_index=True)

jp_stock_m_raw.tail()

,Stock Index / Bond-related Instrument,Stock Index,Time,Log Monthly Return,Volume
328,iShares Core Japan Government Bond ETF,2561.T,2025-06,0.007204,282937
329,iShares Core Japan Government Bond ETF,2561.T,2025-07,-0.016283,276779
330,iShares Core Japan Government Bond ETF,2561.T,2025-08,-0.004130,178697
331,iShares Core Japan Government Bond ETF,2561.T,2025-09,0.003209,273959
332,iShares Core Japan Government Bond ETF,2561.T,2025-10,-0.000916,691174


##### **4.5) REAL GDP**

In [99]:
# Real Gross Domestic Product for Japan (quarterly, billions of chained 2015 JPY, seasonally adjusted, 1994-01, 2025-04)
# https://fred.stlouisfed.org/series/JPNRGDPEXP

jp_rgdp_m_raw = fetch_FRED("JPNRGDPEXP") 
jp_rgdp_m_raw = jp_rgdp_m_raw.rename(columns= 
        {"date": "Time", 
         "JPNRGDPEXP": "Real GDP (billions chained 2015 JPY)"
})
jp_rgdp_m_raw["Country"] = "JP"

# Increasing data granularity from quarterly to monthly data by extending the quarter value to single months 
jp_rgdp_m_raw["Time"] = pd.PeriodIndex(jp_rgdp_m_raw["Time"], freq="Q").to_timestamp()
expanded_rows = []

for _, row in jp_rgdp_m_raw.iterrows():
    quarter_end = row["Time"]
    start_month = quarter_end - pd.offsets.QuarterEnd(startingMonth=3) + pd.DateOffset(days=1)
    for i in range(3):
        month = (start_month + pd.DateOffset(months=i)).strftime("%Y-%m")
        expanded_rows.append({
            "Country": row["Country"],
            "Time": month,
            "Real GDP (billions chained 2015 JPY)": row["Real GDP (billions chained 2015 JPY)"] 
        })

jp_rgdp_m_raw = pd.DataFrame(expanded_rows)
jp_rgdp_m_raw["Country"] = "JP"
jp_rgdp_m_raw = jp_rgdp_m_raw[["Country", "Time", "Real GDP (billions chained 2015 JPY)"]]

jp_rgdp_m_raw.tail()

,Country,Time,Real GDP (billions chained 2015 JPY)
379,JP,2025-08,589417.7
380,JP,2025-09,589417.7
381,JP,2025-10,589727.6
382,JP,2025-11,589727.6
383,JP,2025-12,589727.6


##### **4.6) LONG-TERM GOVERNMENT BOND YIELDS - 10-YEAR**

In [100]:
# Interest Rates: Long-Term Government Bond Yields: 10-Year: Main (Including Benchmark) for Japan (monthly, percent, non-seasonally adjusted, 1989-01, 2025-09)
# https://fred.stlouisfed.org/series/IRLTLT01JPM156N

jp_10ygb_m_raw = fetch_FRED("IRLTLT01JPM156N") 
jp_10ygb_m_raw = jp_10ygb_m_raw.rename(columns= 
        {"date": "Time", 
         "IRLTLT01JPM156N": "10-Year Gov Bond Yields (%)"
})

jp_10ygb_m_raw["Time"] = jp_10ygb_m_raw["Time"].dt.to_period("M").astype(str)

jp_10ygb_m_raw["Country"] = "JP"
jp_10ygb_m_raw = jp_10ygb_m_raw[["Country", "Time", "10-Year Gov Bond Yields (%)"]]

jp_10ygb_m_raw.tail()

,Country,Time,10-Year Gov Bond Yields (%)
440,JP,2025-09,1.645
441,JP,2025-10,1.655
442,JP,2025-11,1.805
443,JP,2025-12,2.060
444,JP,2026-01,2.240


##### **4.7) IMMEDIATE RATES (< 24 Hours): CALL MONEY/INTERBANK RATE**

In [101]:
# Interest Rates: Immediate Rates (< 24 Hours): Call Money/Interbank Rate: Total for Japan (monthly, percent, non-seasonally adjusted, 1985-01, 2025-09)
# https://fred.stlouisfed.org/series/IRSTCI01JPM156N

jp_cmibr_m_raw = fetch_FRED("IRSTCI01JPM156N") 
jp_cmibr_m_raw = jp_cmibr_m_raw.rename(columns= 
        {"date": "Time", 
         "IRSTCI01JPM156N": "Call Money/Interbank Immediate (%)"
})

jp_cmibr_m_raw["Time"] = jp_cmibr_m_raw["Time"].dt.to_period("M").astype(str)

jp_cmibr_m_raw["Country"] = "JP"
jp_cmibr_m_raw = jp_cmibr_m_raw[["Country", "Time", "Call Money/Interbank Immediate (%)"]]

jp_cmibr_m_raw.tail()

,Country,Time,Call Money/Interbank Immediate (%)
482,JP,2025-09,0.477
483,JP,2025-10,0.477
484,JP,2025-11,0.478
485,JP,2025-12,0.557
486,JP,2026-01,0.728


##### **4.8) NATURAL RATE OF INTEREST: SHORT-TERM (1-year) & LONG-TERM (10-year)**

In [102]:
# Natural rate of interest: Short-term (1-year) and long-term (10-year) natural rate of interest (mean estimates and 95% confidence intervals, total for Japan (quarterly, percent, non-seasonally adjusted, 1995-01, 2025-12))
# https://sites.google.com/site/jnakajimaweb/rstar
# Nakajima, J., N. Sudo, Y. Hogen, and Y. Takizuka (2023). "On the estimation of the natural yield curve" Discussion Paper Series A.753, Institute of Economic Research, Hitotsubashi University
# (!!!) Quarterly data, 1Y_Mean and 10Y_Mean

jp_nir_m_raw = pd.read_csv("Data/nakajima et al. (2023).csv")

# (!!!) The dataset is built missing column headlines, so we have to restructure it 
# (!!!) We take only the average values
jp_nir_m_raw = jp_nir_m_raw.iloc[1:].reset_index(drop=True) 
jp_nir_m_raw = jp_nir_m_raw.rename(columns={
    jp_nir_m_raw.columns[0]: "YYYYQ",
    jp_nir_m_raw.columns[1]: "1Y_Mean",
    jp_nir_m_raw.columns[4]: "10Y_Mean"
})

# We need to isolate year and quarter 
jp_nir_m_raw["Year"] = jp_nir_m_raw["YYYYQ"].astype(str).str[:4].astype(int)
jp_nir_m_raw["Quarter"] = jp_nir_m_raw["YYYYQ"].astype(str).str[-1].astype(int)
# Quarter start month
quarter_start_month = {1: 1, 2: 4, 3: 7, 4: 10}
jp_nir_m_raw["StartMonth"] = jp_nir_m_raw["Quarter"].map(quarter_start_month)

# Quarter start date
jp_nir_m_raw["QuarterStart"] = pd.to_datetime(
    dict(year=jp_nir_m_raw["Year"], month=jp_nir_m_raw["StartMonth"], day=1),
    errors="coerce"
)

# Expansion of quarterly data to monthly (flat within quarter)
jp_nir_m_raw = (
    jp_nir_m_raw
    .loc[jp_nir_m_raw.index.repeat(3)]
    .assign(
        Time=lambda x: x["QuarterStart"] + x.groupby(level=0).cumcount().map(lambda i: pd.DateOffset(months=i))
    )
)
jp_nir_m_raw["Time"] = pd.to_datetime(jp_nir_m_raw["Time"], errors="coerce")
jp_nir_m_raw = (
    jp_nir_m_raw
    .assign(
        Time=lambda x: x["Time"].dt.to_period("M").astype(str),
        Country="JP"
    )
    [["Country", "Time", "1Y_Mean", "10Y_Mean"]]
    .reset_index(drop=True)
)
jp_nir_m_raw = jp_nir_m_raw.rename(columns={
    "1Y_Mean": "Est. 1-year Neutral Interest Rate (%)",
    "10Y_Mean": "Est. 10-year Neutral Interest Rate (%)"
})

jp_nir_m_raw.tail()

,Country,Time,Est. 1-year Neutral Interest Rate (%),Est. 10-year Neutral Interest Rate (%)
367,JP,2025-08,-0.27,0.481
368,JP,2025-09,-0.27,0.481
369,JP,2025-10,-0.258,0.497
370,JP,2025-11,-0.258,0.497
371,JP,2025-12,-0.258,0.497


##### **4.9) MARKET YIELD ON U.S. TREASURY SECURITY AT 10-Year CONSTANT MATURITY**

In [103]:
# Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity, Quoted on an Investment Basis (daily, percent, non-seasonally adjusted, 1962-01, 2026-01)
# https://fred.stlouisfed.org/series/DGS10

jp_ust_m_raw = fetch_FRED("DGS10") 
jp_ust_m_raw = jp_ust_m_raw.rename(columns= 
        {"date": "Time", 
         "DGS10": "10-Year US T-Bills Yield (%)"
})
jp_ust_m_raw["Time"] = pd.to_datetime(jp_ust_m_raw["Time"])
jp_ust_m_raw = (
    jp_ust_m_raw
    .set_index("Time")
    .resample("M")
    .mean()
    .reset_index()
)
jp_ust_m_raw["Time"] = jp_ust_m_raw["Time"].dt.to_period("M").astype(str)

jp_ust_m_raw["Country"] = "JP"
jp_ust_m_raw = jp_ust_m_raw[["Country", "Time", "10-Year US T-Bills Yield (%)"]]

jp_ust_m_raw.tail()

,Country,Time,10-Year US T-Bills Yield (%)
766,JP,2025-11,4.093889
767,JP,2025-12,4.143182
768,JP,2026-01,4.213500
769,JP,2026-02,4.125789
770,JP,2026-03,4.050000


##### **4.10) CBOE VOLATILITY INDEX: VIX**

In [104]:
# CBOE Volatility Index: VIX (daily, index, non-seasonally adjusted, 1990-01, 2026-01)
# https://fred.stlouisfed.org/series/VIXCLS

jp_vix_m_raw = fetch_FRED("VIXCLS") 
jp_vix_m_raw = jp_vix_m_raw.rename(columns= 
        {"date": "Time", 
         "VIXCLS": "CBOE-VIX"
})
jp_vix_m_raw["Time"] = pd.to_datetime(jp_vix_m_raw["Time"])
jp_vix_m_raw = (
    jp_vix_m_raw
    .set_index("Time")
    .resample("M")
    .mean()
    .reset_index()
)
jp_vix_m_raw["Time"] = jp_vix_m_raw["Time"].dt.to_period("M").astype(str)

jp_vix_m_raw["Country"] = "JP"
jp_vix_m_raw = jp_vix_m_raw[["Country", "Time", "CBOE-VIX"]]

jp_vix_m_raw.tail()

,Country,Time,CBOE-VIX
430,JP,2025-11,19.769500
431,JP,2025-12,15.548182
432,JP,2026-01,16.179048
433,JP,2026-02,19.207000
434,JP,2026-03,22.505000


##### **4.11) CENTRAL GOVERNMENT DEBT (% of GDP)**

In [105]:
# Central government debt, total (% of GDP) for Japan (annual, % of GDP, non-seasonally adjusted, 1990, 2022)
# https://fred.stlouisfed.org/series/DEBTTLJPA188A 

jp_cgdebt_m_raw = fetch_FRED("DEBTTLJPA188A")
jp_cgdebt_m_raw = jp_cgdebt_m_raw.rename(columns={
    "date": "Time",
    "DEBTTLJPA188A": "Central Government Debt (% GDP)"
})
jp_cgdebt_m_raw["Time"] = pd.to_datetime(jp_cgdebt_m_raw["Time"])
jp_cgdebt_m_raw = (
    jp_cgdebt_m_raw
    .set_index("Time")
    .resample("MS")        
    .ffill()              
    .reset_index()
)
jp_cgdebt_m_raw["Time"] = jp_cgdebt_m_raw["Time"].dt.to_period("M").astype(str)
jp_cgdebt_m_raw["Country"] = "JP"
jp_cgdebt_m_raw = jp_cgdebt_m_raw[
    ["Country", "Time", "Central Government Debt (% GDP)"]
]

jp_cgdebt_m_raw.tail()

,Country,Time,Central Government Debt (% GDP)
380,JP,2021-09,216.135005
381,JP,2021-10,216.135005
382,JP,2021-11,216.135005
383,JP,2021-12,216.135005
384,JP,2022-01,215.906360


### **5) DATA MERGING**

In [106]:
# Policy Rate Dataframe
jp_policy_rate_df = pd.merge(jp_10ygb_m_raw, jp_cmibr_m_raw, on=["Country", "Time"], how="outer")
jp_policy_rate_df = pd.merge(jp_policy_rate_df, jp_nir_m_raw, on=["Country", "Time"], how="outer")

jp_policy_rate_df.to_csv("Data/Aggregated/jp_policy_rate_df.csv", index=False)
jp_policy_rate_df.tail()

,Country,Time,10-Year Gov Bond Yields (%),Call Money/Interbank Immediate (%),Est. 1-year Neutral Interest Rate (%),Est. 10-year Neutral Interest Rate (%)
482,JP,2025-09,1.645,0.477,-0.27,0.481
483,JP,2025-10,1.655,0.477,-0.258,0.497
484,JP,2025-11,1.805,0.478,-0.258,0.497
485,JP,2025-12,2.060,0.557,-0.258,0.497
486,JP,2026-01,2.240,0.728,NaN,NaN


In [107]:
# Exchange Rate Dataframe
jp_exchange_rate_df = pd.merge(REXUSDJPY_m_raw, JPYUSD_m_raw, on=["Country", "Time"], how="outer")

jp_exchange_rate_df.to_csv("Data/Aggregated/jp_exchange_rate_df.csv", index=False)
jp_exchange_rate_df.tail()

,Country,Time,USD-JPY reer CPI-based (Index 2015=100),JPY-USD Spot Exchange Rate
668,JP,2025-09,80.85533,147.8629
669,JP,2025-10,79.29526,151.3545
670,JP,2025-11,77.64572,155.1411
671,JP,2025-12,76.62940,155.9150
672,JP,2026-01,75.62581,156.6505


In [108]:
# Inflation Dataframe
jp_inflation_df = jp_HICP_m_raw.copy()

jp_inflation_df.to_csv("Data/Aggregated/jp_inflation_df.csv", index=False)
jp_inflation_df.tail()

,Country,Time,HICP (NSA)
659,JP,2024-12,110.7
660,JP,2025-01,111.2
661,JP,2025-02,110.8
662,JP,2025-03,111.1
663,JP,2025-04,111.5


In [109]:
# Real GDP Dataframe
jp_rgdp_df = jp_rgdp_m_raw.copy()

jp_rgdp_df.to_csv("Data/Aggregated/jp_rgdp_df.csv", index=False)
jp_rgdp_df.tail()

,Country,Time,Real GDP (billions chained 2015 JPY)
379,JP,2025-08,589417.7
380,JP,2025-09,589417.7
381,JP,2025-10,589727.6
382,JP,2025-11,589727.6
383,JP,2025-12,589727.6


In [110]:
# Stock Indeces Dataframe
jp_stocks_pv_df = jp_stock_m_raw.copy()

jp_stocks_pv_df.to_csv("Data/Aggregated/jp_stocks_pv_df.csv", index=False)
jp_stocks_pv_df.tail()

,Stock Index / Bond-related Instrument,Stock Index,Time,Log Monthly Return,Volume
328,iShares Core Japan Government Bond ETF,2561.T,2025-06,0.007204,282937
329,iShares Core Japan Government Bond ETF,2561.T,2025-07,-0.016283,276779
330,iShares Core Japan Government Bond ETF,2561.T,2025-08,-0.004130,178697
331,iShares Core Japan Government Bond ETF,2561.T,2025-09,0.003209,273959
332,iShares Core Japan Government Bond ETF,2561.T,2025-10,-0.000916,691174


In [111]:
# Control Dataframe
jp_control_df = pd.merge(jp_ust_m_raw, jp_vix_m_raw, on=["Country", "Time"], how="outer")

jp_control_df.to_csv("Data/Aggregated/jp_control_df.csv", index=False)
jp_control_df.tail()

,Country,Time,10-Year US T-Bills Yield (%),CBOE-VIX
766,JP,2025-11,4.093889,19.769500
767,JP,2025-12,4.143182,15.548182
768,JP,2026-01,4.213500,16.179048
769,JP,2026-02,4.125789,19.207000
770,JP,2026-03,4.050000,22.505000


In [112]:
# Aggregated JP Dataframe
jp_aggregated_df = pd.merge(jp_policy_rate_df, jp_exchange_rate_df, on=["Country", "Time"], how="outer")
jp_aggregated_df = pd.merge(jp_aggregated_df, jp_inflation_df, on=["Country", "Time"], how="outer")
jp_aggregated_df = pd.merge(jp_aggregated_df, jp_rgdp_df, on=["Country", "Time"], how="outer")
jp_aggregated_df = pd.merge(jp_aggregated_df, jp_control_df, on=["Country", "Time"], how="outer")

jp_aggregated_df.to_csv("Data/Aggregated/jp_aggregated_df.csv", index=False)
jp_aggregated_df.tail()

,Country,Time,10-Year Gov Bond Yields (%),Call Money/Interbank Immediate (%),Est. 1-year Neutral Interest Rate (%),Est. 10-year Neutral Interest Rate (%),USD-JPY reer CPI-based (Index 2015=100),JPY-USD Spot Exchange Rate,HICP (NSA),Real GDP (billions chained 2015 JPY),10-Year US T-Bills Yield (%),CBOE-VIX
766,JP,2025-11,1.805,0.478,-0.258,0.497,77.64572,155.1411,NaN,589727.6,4.093889,19.769500
767,JP,2025-12,2.060,0.557,-0.258,0.497,76.62940,155.9150,NaN,589727.6,4.143182,15.548182
768,JP,2026-01,2.240,0.728,NaN,NaN,75.62581,156.6505,NaN,NaN,4.213500,16.179048
769,JP,2026-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.125789,19.207000
770,JP,2026-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.050000,22.505000
